In [ ]:
import os

os.environ["TORCHINDUCTOR_CACHE_DIR"] = "torch_cache"
os.environ["TORCHINDUCTOR_FX_GRAPH_CACHE"] = "1"
os.environ["TORCHINDUCTOR_AUTOGRAD_CACHE"] = "1"

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import cv2
import numpy as np
import torch
import gc
import time
import torch
from realesrgan.archs.rrdb_pixelshuffle_v3_arch import RRDBNet_v3 as RRDBNet


torch._dynamo.config.recompile_limit = float('inf')  # always allow recompile to support many image sizes
torch._dynamo.config.accumulated_recompile_limit = float('inf') 


In [ ]:
pad = 5
scale = 4
device = "cuda"
#device = "cpu"

In [ ]:

max_size4x = 2100*2100

allowed_resolutions_x4 = []
for y in range(1024,6500,256):
    for x in range(1024,6500,256):
        allowed_resolutions_x4.append((x,y))

# apply padding 
allowed_resolutions_x4_post_pad = []
for i in allowed_resolutions_x4:
    new_size = (i[0]+pad*2, i[1]+pad*2)
    if new_size[0] * new_size[1] <= max_size4x:
        allowed_resolutions_x4_post_pad.append(new_size)
    else:
        #print(new_size, "is too large", (new_size[0] * new_size[1]))
        pass

sorted(allowed_resolutions_x4_post_pad, key = lambda x: x[0]*x[1])

In [ ]:
#loadnet = torch.load("../Real-ESRGAN/experiments/train_v3_attn_x4_m/models/net_g_35000.pth")
#if 'params_ema' in loadnet:
#    keyname = 'params_ema'
#else:
#    keyname = 'params'
#print(keyname)

model_4x = RRDBNet(num_in_ch=3, num_out_ch=3, highway_channels=512, processing_channels=128, num_block=32, num_grow_ch=64, num_pre_upscale_ch = 256, scale=scale, use_attention=True)
_ = model_4x.eval()
#model_4x.load_state_dict(loadnet[keyname], strict=True)

model_4x_compile = RRDBNet(num_in_ch=3, num_out_ch=3, highway_channels=512, processing_channels=128, num_block=32, num_grow_ch=64, num_pre_upscale_ch = 256, scale=scale, use_attention=True)
_ = model_4x_compile.eval()
#model_4x_compile.load_state_dict(loadnet[keyname], strict=True)
model_4x_compile.compile(dynamic=False, fullgraph=True)

In [ ]:
def find_best_resolution(w, h, resolutions):
    """
    Find the smallest resolution from the list that can fit the input dimensions.
    Returns (target_w, target_h) and the padding needed.
    """
    for target_w, target_h in sorted(resolutions, key = lambda x: x[0]*x[1]):
        if w <= target_w and h <= target_h:
            # Calculate padding needed (equally distributed on all sides)
            pad_w = target_w - w
            pad_h = target_h - h
            
            # Distribute padding equally, with extra pixel going to right/bottom if odd
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left
            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            
            return (target_w, target_h), (pad_left, pad_right, pad_top, pad_bottom)
    
    return None

def _should_use_compiled(h,w,scale):
    if scale == 4:
        return w*h > 1600**2
        
    return False

def process(img, scale):

    t0 = time.time()
    
    img = img.astype(np.float32)
    if np.max(img) > 256:  # 16-bit image
        max_range = 65535
        print('\tInput is a 16-bit image')
    else:
        max_range = 255
    img = img / max_range
    img = torch.from_numpy(np.transpose(img, (2, 0, 1))).float()
    img = img.unsqueeze(0)

    # pre_pad
    img = torch.nn.functional.pad(img, (5, 5, 5, 5), 'reflect').float()
    
    # mod scale
    mod_scale = 1
    if scale == 2:
        mod_scale = 2
    if scale == 1:
        mod_scale = 4
    
    mod_pad_h, mod_pad_w = 0, 0
    b, c, h, w = img.size()
    if (h % mod_scale != 0):
      mod_pad_h = (mod_scale - h % mod_scale)
    if (w % mod_scale != 0):
        mod_pad_w = (mod_scale - w % mod_scale)
    img = torch.nn.functional.pad(img, (0, mod_pad_w, 0, mod_pad_h), 'reflect')

    b, c, h, w = img.size()
    should_use_compiled = _should_use_compiled(h,w,scale)

    print(w, h, "- compile:", should_use_compiled)

    if should_use_compiled:
        # Fixed resolution padding
        b, c, h, w = img.size()
        print(f"Image size after pre-padding: {w}x{h}")
    
        if scale == 4:
            target_resolutions = allowed_resolutions_x4_post_pad

        res = find_best_resolution(w, h, target_resolutions)
        if res is None:
            return None
        target_resolution, (pad_left, pad_right, pad_top, pad_bottom) = res
        print(f"Selected resolution: {target_resolution[0]}x{target_resolution[1]}")
        print(f"Resolution padding: left={pad_left}, right={pad_right}, top={pad_top}, bottom={pad_bottom}")
    
        # Apply resolution padding (left, right, top, bottom)
        resolution_pad = (pad_left, pad_right, pad_top, pad_bottom)
        img = torch.nn.functional.pad(img, resolution_pad, 'reflect')
    
    
    if scale == 4 and should_use_compiled:
        selected_model = model_4x_compile
    if scale == 4 and not should_use_compiled:
        selected_model = model_4x

    selected_model.to(device)
    img = img.to(device)
    
    print("testing...")
    with torch.no_grad(): 
        with torch.amp.autocast(device):
            out = selected_model(img)

    selected_model.to("cpu")

        
    # Remove resolution padding first (in reverse order)
    print(out.size())
    
    if should_use_compiled:
        if any(p > 0 for p in resolution_pad):
            _, _, h, w = out.size()
            # Scale the padding by the upscale factor
            out_pad_left = pad_left * scale
            out_pad_right = pad_right * scale
            out_pad_top = pad_top * scale
            out_pad_bottom = pad_bottom * scale
            
            out = out[:, :, out_pad_top:h - out_pad_bottom, out_pad_left:w - out_pad_right]
            print(f"Removed resolution padding: {out_pad_left}, {out_pad_right}, {out_pad_top}, {out_pad_bottom}")


    # remove mod pad
    if mod_scale is not None:
        _, _, h, w = out.size()
        out = out[:, :, 0:h - mod_pad_h * scale, 0:w - mod_pad_w * scale]
    
    
    # remove prepad
    _, _, h, w = out.size()
    out = out[:, :, 5*scale:h - 5*scale, 5*scale:w - 5*scale]
    
    
    output_img = out.data.squeeze().float().cpu().clamp_(0, 1).numpy()
    output_img = np.transpose(output_img[:, :, :], (1, 2, 0))
    
    if max_range == 65535:  # 16-bit image
        output_img = (output_img * 65535.0).round().astype(np.uint16)
    else:
        output_img = (output_img * 255.0).round().astype(np.uint8)

  
    print(time.time() - t0)
    
    return output_img
    


In [ ]:
img = cv2.imread("tests/data/lq_4/baboon.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (2048, 2048))
print(img.shape)
out = process(img, 4)


In [ ]:
import matplotlib.pyplot as plt
plt.imshow(out)

In [ ]:
#for x in [200, 1000,2000,3000,4100]:
#    for y in [500,600,1200,3100, 4100]:
for x in [200, 1000,2000]:
    for y in [500,600,1200]:
#for x in [3000,4000]:
#    for y in [3000,4000, 3500,3100,3200,3300]:

        img = cv2.imread("tests/data/lq_4/baboon.png")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (x,y))
        print (img.shape)
        out = process(img, scale)

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()